In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# default_exp unsupervised_learning

# chemcluster

> Perform clustering using Kmeans, HDBSCAN or Butina.

In [ ]:
# export

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from collections import defaultdict
from tqdm.auto import tqdm
import optuna

from fastcore.basics import *
from fastcore.foundation import *
from fastcore.meta import *
from chemcluster.typing_basics import *

from sklearn.cluster import KMeans
from rdkit.ML.Cluster import Butina
from sklearn.metrics import silhouette_score, silhouette_samples
import hdbscan

from rdkit import Chem
from rdkit import DataStructs
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem import MACCSkeys
from rdkit.Chem.AtomPairs import Pairs

from kneed import KneeLocator

In [ ]:
#hide
from nbdev.showdoc import show_doc

In [ ]:
# export
class BaseClustering:
    
    """Base class to perform clustering on a collection of molecules. 
    Use children classes `KMeansClustering`, `HDBSCANClustering`, `ButinaClustering` to cluster molecules
    
    """
    
    def __init__(self):
        pass
    
    def cluster(self):
        pass
                
    @property
    def clusterer(self):
        return self._clusterer
    
    @clusterer.setter
    def clusterer(self, i):
        self._clusterer = i
        
    @property
    def labels(self):
        return self._labels
    
    @labels.setter
    def labels(self, i):
        if isinstance(i, Sequence) and not isinstance(i, str):
            self._labels = i

In [ ]:
show_doc(BaseClustering)

<h2 id="BaseClustering" class="doc_header"><code>class</code> <code>BaseClustering</code><a href="" class="source_link" style="float:right">[source]</a></h2>

> <code>BaseClustering</code>()

Base class to perform clustering on a collection of molecules. 
Use children classes [`KMeansClustering`](/chemcluster/clustering.html#KMeansClustering), [`HDBSCANClustering`](/chemcluster/clustering.html#HDBSCANClustering), [`ButinaClustering`](/chemcluster/clustering.html#ButinaClustering) to cluster molecules

In [ ]:
#export  
class KMeansClustering(BaseClustering):
    
    """Performs k-means clustering on a dataset of molecules
    """
    
    def __init__(self, dataset : np.array):
        self.dataset = dataset
            
    def cluster(self, n_clusters:int=10, **kwargs):
        """Run k-means on the dataset
        
        Arguments:
        
        -------------------------------------------       
                n_clusters : int (default=10)
                    Number of clusters

        Keyword arguments:
        
        -------------------------------------------       
                max_iter : int (default=5)
                n_init : int (default=5)
                init : str (default='k-means++')
                random_state : int (default=None)
            
            
        Returns:
        
        -------------------------------------------       
                labels : np.array
                    Clustering labels
        
        """
        
        max_iter = kwargs.get('max_iter', 500)
        n_init = kwargs.get('n_init', 10)
        init = kwargs.get('init', 'k-means++')
        random_state = kwargs.get('random_state', None)
        
        cls = KMeans(n_clusters=n_clusters, init=init, n_init=n_init, max_iter=max_iter, random_state=random_state)
        cls.fit(self.dataset)
        
        self._clusterer = cls
        self._labels = cls.labels_
        return self._labels
    
    def elbow_method(self, n_clusters:List, figsize:Tuple=(12,9), **kwargs):

        self.inertias = []
        for n in n_clusters:
            self.cluster(n, **kwargs)   
            inertia = self.clusterer.inertia_
            self.inertias.append(inertia)
            
        # Find elbow
        params = {"curve": "convex",
                    "direction": "decreasing"}
        
        knee_finder = KneeLocator(n_clusters, self.inertias, **params)
        self.elbow_value = knee_finder.elbow
        
        # Plot Elbow
        sns.set_context('paper',font_scale=2.0)
        sns.set_style('whitegrid')
        
        fig = plt.figure(figsize=figsize)
        ax=sns.lineplot(x=n_clusters, y=np.array(self.inertias), linewidth=2.5, marker='o', color='blue', markersize=7)

        ax.set_xlabel('Number of clusters (K)')
        ax.set_ylabel('Distortion')
        sns.despine(right=True,top=True)
        plt.title('K-means Elbow method',fontweight='bold',fontsize=22)
        
        if self.elbow_value is not None:
            elbow_label = "Elbow at $K={}$".format(self.elbow_value)      
            ax.axvline(self.elbow_value, c='k', linestyle="--",label=elbow_label)
            ax.legend(loc="best", fontsize=18, frameon=True)
        for i in ax.spines.items():
            i[1].set_linewidth(1.5)
            i[1].set_color('k')

        plt.tight_layout()
        plt.show()

In [ ]:
show_doc(KMeansClustering)

<h2 id="KMeansClustering" class="doc_header"><code>class</code> <code>KMeansClustering</code><a href="" class="source_link" style="float:right">[source]</a></h2>

> <code>KMeansClustering</code>(**`dataset`**:`array`) :: [`BaseClustering`](/chemcluster/clustering.html#BaseClustering)

Performs k-means clustering on a dataset of molecules
    

In [ ]:
show_doc(KMeansClustering.cluster, name='KMeansClustering.cluster')

<h4 id="KMeansClustering.cluster" class="doc_header"><code>KMeansClustering.cluster</code><a href="__main__.py#L10" class="source_link" style="float:right">[source]</a></h4>

> <code>KMeansClustering.cluster</code>(**`n_clusters`**:`int`=*`10`*, **\*\*`kwargs`**)

Run k-means on the dataset

Arguments:

-------------------------------------------       
        n_clusters : int (default=10)
            Number of clusters

Keyword arguments:

-------------------------------------------       
        max_iter : int (default=5)
        n_init : int (default=5)
        init : str (default='k-means++')
        random_state : int (default=None)
    
    
Returns:

-------------------------------------------       
        labels : np.array
            Clustering labels

In [ ]:
#export      
class HDBSCANClustering(BaseClustering):
    """Performs HDBSCAN clustering on a dataset of molecules"""
    def __init__(self, dataset : np.array):
        self.dataset = dataset
            
            
    def cluster(self, min_cluster_size:int=5, min_samples:int=None, metric:str='jaccard', **kwargs):
        """Run HDBSCAN clustering on the dataset
        
        Arguments:
        
        ----------------------------------------------------------------------------------------

               min_cluster_size : int, optional (default=5)
                   The minimum size of clusters; single linkage splits that contain
                   fewer points than this will be considered points "falling out" of a
                   cluster rather than a cluster splitting into two new clusters.

               min_samples : int, optional (default=None)
                   The number of samples in a neighbourhood for a point to be
                   considered a core point.

               metric : string, or callable, optional (default='euclidean')
                   The metric to use when calculating distance between instances in a
                   feature array. If metric is a string or callable, it must be one of
                   the options allowed by metrics.pairwise.pairwise_distances for its
                   metric parameter.
                   If metric is "precomputed", X is assumed to be a distance matrix and
                   must be square.
                              
        Keyword arguments:
        
        ----------------------------------------------------------------------------------------
                See HDBSCAN documentation (https://hdbscan.readthedocs.io/en/latest/index.html)
            
        Returns:
        
        ----------------------------------------------------------------------------------------
                labels : np.array
                    Clustering labels
        
        """
        cls = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples, metric=metric, **kwargs)
        cls.fit(self.dataset)
        
        self._clusterer = cls
        self._labels = cls.labels_
        return self._labels
    
    @staticmethod
    def validate_clustering(X, labels):
        return hdbscan.validity_index(X,labels)
    
    

In [ ]:
show_doc(HDBSCANClustering)

<h2 id="HDBSCANClustering" class="doc_header"><code>class</code> <code>HDBSCANClustering</code><a href="" class="source_link" style="float:right">[source]</a></h2>

> <code>HDBSCANClustering</code>(**`dataset`**:`array`) :: [`BaseClustering`](/chemcluster/clustering.html#BaseClustering)

Performs HDBSCAN clustering on a dataset of molecules

In [ ]:
show_doc(HDBSCANClustering.cluster, name='HDBSCANClustering.cluster')

<h4 id="HDBSCANClustering.cluster" class="doc_header"><code>HDBSCANClustering.cluster</code><a href="__main__.py#L8" class="source_link" style="float:right">[source]</a></h4>

> <code>HDBSCANClustering.cluster</code>(**`min_cluster_size`**:`int`=*`5`*, **`min_samples`**:`int`=*`None`*, **`metric`**:`str`=*`'jaccard'`*, **\*\*`kwargs`**)

Run HDBSCAN clustering on the dataset

Arguments:

----------------------------------------------------------------------------------------

       min_cluster_size : int, optional (default=5)
           The minimum size of clusters; single linkage splits that contain
           fewer points than this will be considered points "falling out" of a
           cluster rather than a cluster splitting into two new clusters.

       min_samples : int, optional (default=None)
           The number of samples in a neighbourhood for a point to be
           considered a core point.

       metric : string, or callable, optional (default='euclidean')
           The metric to use when calculating distance between instances in a
           feature array. If metric is a string or callable, it must be one of
           the options allowed by metrics.pairwise.pairwise_distances for its
           metric parameter.
           If metric is "precomputed", X is assumed to be a distance matrix and
           must be square.
                      
Keyword arguments:

----------------------------------------------------------------------------------------
        See HDBSCAN documentation (https://hdbscan.readthedocs.io/en/latest/index.html)
    
Returns:

----------------------------------------------------------------------------------------
        labels : np.array
            Clustering labels

In [ ]:
#export    
class ButinaClustering(BaseClustering):
    """Performs Butina clustering
    
    See original publication at: https://github.com/PatWalters/clusterama
    

    
    """
    def __init__(self, dataset, fp_type="rdkit"):
        self.dataset = dataset
        self.fp_type = fp_type

    def cluster(self,sim_cutoff=0.8):
        """Run Butina clustering on the dataset
        
        Arguments:
        
        ----------------------------------------------------------------------------------------

               sim_cutoff : float, optional (default=0.8)
                   The minimum Tanimoto similarity to consider for putting compounds in the same cluster
            
        Returns:
        
        ----------------------------------------------------------------------------------------
                labels : np.array
                    Clustering labels
        
        """        
        
        
        mol_list = [Chem.MolFromSmiles(x) for x in tqdm(self.dataset,desc="Calculating Fingerprints")]
        return self.cluster_mols(mol_list, sim_cutoff)

    def get_fps(self, mol_list):
        fp_dict = {
            "morgan2" : [AllChem.GetMorganFingerprintAsBitVect(x,2) for x in mol_list],
            "rdkit" : [Chem.RDKFingerprint(x) for x in mol_list],
            "maccs" : [MACCSkeys.GenMACCSKeys(x) for x in mol_list],
            "ap" : [Pairs.GetAtomPairFingerprint(x) for x in mol_list]
            }
        if fp_dict.get(self.fp_type) is None:
            raise KeyError(f"No fingerprint method defined for {self.fp_type}")

        return fp_dict[self.fp_type]
    
    def cluster_mols(self, mol_list, sim_cutoff:float=0.8):
        dist_cutoff = 1.0 - sim_cutoff
        fp_list = self.get_fps(mol_list)
        dists = []
        nfps = len(fp_list)
        for i in range(1, nfps):
            sims = DataStructs.BulkTanimotoSimilarity(fp_list[i],fp_list[:i])
            dists.extend([1-x for x in sims])
        mol_clusters = Butina.ClusterData(dists,nfps,dist_cutoff,isDistData=True)
        cluster_id_list = [0]*nfps
        for idx,cluster in enumerate(mol_clusters,1):
            for member in cluster:
                cluster_id_list[member] = idx
        self._labels = [x-1 for x in cluster_id_list]
        return self._labels

In [ ]:
show_doc(ButinaClustering, name='ButinaClustering')

<h2 id="ButinaClustering" class="doc_header"><code>class</code> <code>ButinaClustering</code><a href="" class="source_link" style="float:right">[source]</a></h2>

> <code>ButinaClustering</code>(**`dataset`**, **`fp_type`**=*`'rdkit'`*) :: [`BaseClustering`](/chemcluster/clustering.html#BaseClustering)

Performs Butina clustering

See original publication at: https://github.com/PatWalters/clusterama

In [ ]:
show_doc(ButinaClustering.cluster, name='ButinaClustering.cluster')

<h4 id="ButinaClustering.cluster" class="doc_header"><code>ButinaClustering.cluster</code><a href="__main__.py#L14" class="source_link" style="float:right">[source]</a></h4>

> <code>ButinaClustering.cluster</code>(**`sim_cutoff`**=*`0.8`*)

Run Butina clustering on the dataset

Arguments:

----------------------------------------------------------------------------------------

       sim_cutoff : float, optional (default=0.8)
           The minimum Tanimoto similarity to consider for putting compounds in the same cluster
    
Returns:

----------------------------------------------------------------------------------------
        labels : np.array
            Clustering labels

In [ ]:
# #export 
# @delegates(BaseClustering)
# def objective(self : BaseClustering, trial, **kwargs):
#     """Objective function to optimize clustering hyperparameters"""
    
#     _params = defaultdict()
#     algorithm = kwargs['algorithm']
    
    
#     _params['kmeans'] = {'n_neighbors' : trial.suggest_int('n_neighbors', 5, 50, step=5)}
#     _params['hdbscan'] = {'n_neighbors' : trial.suggest_int('n_neighbors', 5, 50, step=5),
#               'cluster_selection_epsilon' : trial.suggest_uniform('cluster_selection_epsilon', 0.1, 0.5),
#         'min_cluster_size' : trial.suggest_int("min_cluster_size", 2, 30, step=1),
#         'min_samples' : trial.suggest_int("min_samples", 2, 30, step=1)}
        
#     _params['butina'] = {'fp_type':trial.suggest_categorical('fp_type', ["morgan2", "rdkit", "maccs","ap"]),
#                         'sim_cutoff':trial.suggest_uniform('sim_cutoff', 0.1, 0.5)
#                         }
    
    
 
    
    
    
#     score = silhouette_score(embeddings[labels>=0],labels[labels>=0])
#     print(f'Number of unique clusters = {len(np.unique(labels))} - score = {score}')
#     return score
# study = optuna.create_study(
#     direction="maximize",
#     sampler=optuna.samplers.TPESampler(seed=42),
#     pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
# )


In [ ]:
#hide
from nbdev.export import notebook2script
notebook2script('clustering.ipynb')
#from nbdev.export2html import notebook2html; notebook2html()

Converted clustering.ipynb.
